<a href="https://colab.research.google.com/github/ChrisJavier/UIDE_3-WorkGroupProyectsVisu/blob/main/Week2/src/VADDC_CP_W2_E4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Trabajo 2: # Ejercicio práctico clase 2

# ¿Cómo desplegar la policía de Chicago para luchar eficazmente contra la delincuencia?

In [ ]:
import pandas as pd
import numpy  as np
import plotly.graph_objects as go
import branca
import geopandas
import folium
import os

import plotly.express as px

from wordcloud import WordCloud #
from IPython.display import display

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')



Han encontrado un conjunto de datos a disposición de la policía de Chicago de 2017 con información sobre los delitos cometidos en toda la ciudad.

Van a leer y ver su conjunto de datos. Este conjunto de datos se descarga de este sitio web. Contiene los incidentes delictivos denunciados (a excepción de los asesinatos, de los que existen datos para cada víctima) que se produjeron en la ciudad de Chicago en 2017.


In [ ]:
# Define el nombre del archivo comprimido que se descargará
chicago_crime_Data = 'Chicago_crime_data.csv'
boundariesq_beat = 'Boundaries_beat.geojson'

# Descarga el archivo csv

if not os.path.exists(chicago_crime_Data) and not os.path.exists(chicago_crime_Data):
  !wget --timeout=15 --tries=2 'https://raw.githubusercontent.com/ChrisJavier/UIDE_3-WorkGroupProyectsVisu/refs/heads/main/Week2/dataset/Chicago_crime_data.csv' -O '{chicago_crime_Data}'

if not os.path.exists(boundariesq_beat) and not os.path.exists(boundariesq_beat):
  !wget --timeout=15 --tries=2 'https://raw.githubusercontent.com/ChrisJavier/UIDE_3-WorkGroupProyectsVisu/refs/heads/main/Week2/dataset/Boundaries_beat.geojson' -O '{boundariesq_beat}'

# Leer el archivo
df = pd.read_csv(chicago_crime_Data, sep=',', dtype={'ID': object, 'beat_num': object})
df

--2026-03-17 00:33:49--  https://raw.githubusercontent.com/ChrisJavier/UIDE_3-WorkGroupProyectsVisu/refs/heads/main/Week2/dataset/Chicago_crime_data.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 57763135 (55M) [text/plain]
Saving to: ‘Chicago_crime_data.csv’

Chicago_crime_data. 100%[===================>]  55.09M   289MB/s    in 0.2s    

2026-03-17 00:33:51 (289 MB/s) - ‘Chicago_crime_data.csv’ saved [57763135/57763135]



,ID,Case Number,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,...,Ward,Community Area,FBI Code,X Coordinate,Y Coordinate,Year,Updated On,Latitude,Longitude,Location
0,11192233,JB100016,12/31/17 23:58,046XX N ST LOUIS AVE,630,BURGLARY,ATTEMPT FORCIBLE ENTRY,APARTMENT,False,False,...,33.0,14,5,1152214.0,1930694.0,2017,5/4/18 15:51,41.965694,-87.715726,"(41.965693651, -87.715726125)"
1,11196379,JB105867,12/31/17 23:50,024XX N LAKE SHORE DR NB,460,BATTERY,SIMPLE,MOVIE HOUSE/THEATER,False,False,...,43.0,7,08B,1175293.0,1916610.0,2017,5/4/18 15:51,41.926559,-87.631294,"(41.926558908, -87.631294073)"
2,11192540,JB100551,12/31/17 23:48,001XX E SUPERIOR ST,890,THEFT,FROM BUILDING,HOTEL/MOTEL,False,False,...,42.0,8,6,1177508.0,1905401.0,2017,5/4/18 15:51,41.895751,-87.623496,"(41.895750913, -87.623495923)"
3,11192239,JB100032,12/31/17 23:45,019XX S CANAL ST,1320,CRIMINAL DAMAGE,TO VEHICLE,STREET,False,True,...,25.0,31,14,1173432.0,1891037.0,2017,5/4/18 15:51,41.856427,-87.638893,"(41.856426716, -87.638892854)"
4,11192254,JB100003,12/31/17 23:45,115XX S STATE ST,041A,BATTERY,AGGRAVATED: HANDGUN,RESIDENCE,False,True,...,34.0,53,04B,1178329.0,1828012.0,2017,5/4/18 15:51,41.683369,-87.622830,"(41.683369303, -87.622829524)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
268298,11035993,JA367627,1/1/17 0:00,026XX W COYLE AVE,1752,OFFENSE INVOLVING CHILDREN,AGG CRIM SEX ABUSE FAM MEMBER,RESIDENCE,False,False,...,50.0,2,20,1157570.0,1946019.0,2017,2/10/18 15:50,42.007638,-87.695614,"(42.007638503, -87.695613598)"
268299,10942975,JA261045,1/1/17 0:00,035XX S GILES AVE,281,CRIM SEXUAL ASSAULT,NON-AGGRAVATED,"SCHOOL, PUBLIC, BUILDING",True,False,...,2.0,35,2,1178842.0,1881615.0,2017,2/10/18 15:50,41.830450,-87.619323,"(41.830450306, -87.61932306)"
268300,10942796,JA260938,1/1/17 0:00,028XX N WESTERN AVE,1330,CRIMINAL TRESPASS,TO LAND,OTHER,False,False,...,1.0,21,26,1159863.0,1918955.0,2017,2/10/18 15:50,41.933326,-87.687927,"(41.933326413, -87.687927299)"
268301,10801141,JA100083,1/1/17 0:00,011XX W DICKENS AVE,486,BATTERY,DOMESTIC BATTERY SIMPLE,APARTMENT,False,True,...,43.0,7,08B,1168452.0,1914119.0,2017,2/14/17 15:49,41.919874,-87.656504,"(41.919874416, -87.656503702)"


Podemos ver arriba que la tabla contiene 22 columnas y hay 268.303 registros en total. A continuación se ofrece una breve descripción de cada columna:

|     Variable name    |                    Descripción de la variable                    |                                                                                                  Nota                                                                                                  |
|:--------------------:|:---------------------------------------------------------------:|:------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------:|
|          ID          |              Identificador único para el registro              |                                                                   Cada víctima en un solo caso de homicidio se asigna a un ID diferente                                                                  |
|      Case Number     | El número RD (División de Registros) del Departamento de Policía de Chicago |                                                  Único para el incidente. Múltiples IDs pueden compartir el mismo número de caso si el incidente es un caso de homicidio                                                 |
|         Date         |               Fecha en que ocurrió el incidente              |                                                                                Podría ser una mejor estimación para algunos registros                                                                               |
|         Block        |     La dirección parcialmente redactada donde ocurrió el incidente     |                                                                    La dirección redactada está en el mismo bloque que la dirección real                                                                    |
|         IUCR         |          El código de informe uniforme de crímenes de Illinois          |                                Vinculado directamente al tipo principal y la descripción del delito. Ver detalles [aquí](https://data.cityofchicago.org/widgets/c7ck-438e)                                |
|     Primary Type     |          La descripción principal del código IUCR          |      -                                                                                                                                                                                                  |
|      Description     |         La descripción secundaria del código IUCR         |     -                                                                                                                                                                                                   |
| Location Description |   Descripción de la ubicación donde ocurrió el incidente  |    -                                                                                                                                                                                                    |
|        Arrest        |                  Si se realizó un arresto                 |     -                                                                                                                                                                                                   |
|       Domestic       |          Si el incidente estaba relacionado con violencia doméstica         |                                                                La definición de relacionado con violencia doméstica se basa en la Ley de Violencia Doméstica de Illinois                                                               |
|       beat_num       |         El área de patrulla policial donde ocurrió el incidente        |          El área geográfica policial más pequeña - cada área de patrulla tiene un coche policial dedicado. Ver detalles [aquí](https://data.cityofchicago.org/Public-Safety/Boundaries-Police-Beats-current-/aerh-rz74)         |
|       District       |       El distrito policial donde ocurrió el incidente      | Tres a cinco áreas de patrulla conforman un sector policial y tres sectores conforman un distrito policial. Ver detalles [aquí](https://data.cityofchicago.org/Public-Safety/Boundaries-Police-Districts-current-/fthy-xz3r) |
|         Ward         |            La zona donde ocurrió el incidente            |                          Las zonas son distritos del consejo municipal. Ver detalles [aquí](https://data.cityofchicago.org/Facilities-Geographic-Boundaries/Boundaries-Wards-2015-/sp34-6z76)                          |
|    Community Area    |       El área comunitaria donde ocurrió el incidente       |                                     Ver detalles [aquí](https://data.cityofchicago.org/Facilities-Geographic-Boundaries/Boundaries-Community-Areas-current-/cauq-8yn6)                                    |
|       FBI Code       |   La clasificación del crimen según lo descrito en el NIBRS del FBI  |                         NIBRS significa Sistema Nacional de Informes de Incidentes Basados (NIBRS). Ver detalles [aquí](http://gis.chicagopolice.org/clearmap_crime_sums/crime_types.html)                         |
|       Latitude       |  La latitud de la ubicación donde ocurrió el incidente  |                                                   Esta ubicación se desplaza de la ubicación real para redacción parcial pero cae en el mismo bloque                                                  |
|       Longitude      |  La longitud de la ubicación donde ocurrió el incidente |       -

# Configuración de celda para las visualizaciones

In [ ]:
def enable_plotly_in_cell():
  import IPython
  from plotly.offline import init_notebook_mode
  display(IPython.core.display.HTML('''
        <script src="/static/components/requirejs/require.js"></script>
  '''))
  init_notebook_mode(connected=False) # modo offline

get_ipython().events.register('pre_run_cell', enable_plotly_in_cell)

## Fase 1: Investigando crímenes por tipo y descripción.

Para iniciar nuestro análisis vamos a explorar la relación entre las variables "primary_type" y "description". Como ambas variables son discretas podemos contar el número total de registros que pertenecen a categorías específicas. Tenga en cuenta que cada tipo de "primary_type" tiene su propio conjunto de descripciones que no se solapan. Es decir si dos delitos tienen diferentes "primary_type" entonces no tendrán la misma descripción por definición (a esto se le conoce como variables anidadas).

**Tarea 1**: Crea un gráfico que permita observar la cantidad de ocurrencias por delito.

**Pregunta 1**: Qué pueden identificar??

In [ ]:
# Coloca el código aquí ############
# Tarea 1


**RTA**:

Como sabemos que estas dos variables son anidadas podemos desglosar aún más las frecuencias anteriores incluyendo la descripción.

**Tarea 2**: Escriba código utilizando la función groupby, para contar el número de casos en todas las combinaciones de ``Primary_type`` y ``Description``. A continuación, ordene los resultados en orden decreciente según el número de casos, y grafique en un gráfico de barras agrupado o apilados donde el eje x sea el Primary_type y las barras sean la Description.

**Pregunta 2**: ¿Cuáles son las descripciones más frecuentes de casos de robo (THEFT), lesiones (Battery) y daños penales (Criminal Damage) en Chicago?

In [ ]:
data_1 = df.groupby(["Primary Type", "Description"])["ID"].count().reset_index(name="count").sort_values(by="count", ascending=False).reset_index(drop=True)
data_1.head()

In [ ]:
len(data_1.head(20)["Description"].unique())

In [ ]:
# Coloquen el código aquí ####################
# Tarea 2
fig = px.bar(data_1.head(20), x="Primary Type", y="count", color="Description", barmode="group")
fig.update_layout(bargap=.3)
fig.update_traces(width=0.4)
fig.show()

**RTA**:

Del análisis de frecuencias, sabemos que hay 310 descripciones en total y presentarlas todas en el gráfico no es viable. Sin embargo, podemos utilizar una herramienta de visualización conocida como nube de palabras (word cloud) para resumir las descripciones predominantes dentro de cada tipo primario. Una nube de palabras visualiza las palabras dentro de una colección de textos (en nuestro caso, los textos son todas las descripciones de un tipo primario específico) y el tamaño de cada palabra es proporcional a la frecuencia con que aparece en la base de datos. A continuación, construimos tres nubes de palabras para los 3 tipos de delitos primarios más frecuentes:

In [ ]:
# wordcloud para primary type definido por un rango o posición del crimen
def wordcloud_crime( df, rank, word):
    df_filter = df[df["Primary Type"]==df["Primary Type"].value_counts().index[rank]]
    text = ' '.join(df_filter['Description'])
    wordcloud = WordCloud(max_font_size=50, max_words=100, background_color="white").generate(text)
    # usamos en este caso matplotlib
    #plt.imshow(wordcloud)
    fig = px.imshow(wordcloud, binary_string=True)
    fig.update_layout(title=f'Nube de palabras para {word}')
    fig.show()

In [ ]:
word_i = df["Primary Type"].value_counts().index[0]
wordcloud_crime( df, 0, word_i)

In [ ]:
word_i = df["Primary Type"].value_counts().index[1]
wordcloud_crime( df, 1, word_i)

In [ ]:
word_i = df["Primary Type"].value_counts().index[2]
wordcloud_crime( df, 2, word_i)

**Pregunta 3**: ¿Cuáles son las palabras más comunes para describir estos tipos de casos?

**RTA:**

Como hemos visto con las nubes de palabras, parece que un determinado tipo de delito suele estar relacionado con ciertos tipos de lugares (por ejemplo, en casa, en tiendas).

**Tarea 3:** Creen una gráfica para investigar los patrones de delincuencia asociados a los tipos de lugares de los delitos (Location Description).

**Pregunta 4:** Basándote en los resultados, ¿qué tipos de lugares son más propensos a la delincuencia?

In [ ]:
# Coloca el código aquí ############
# Tarea 3


**RTA**:

Hasta ahora, hemos visto patrones delictivos vinculados con el "Tipo principal" y la "Descripción de la ubicación" por separado. Tiene sentido ver si una determinada combinación de tipo de delito y tipo de ubicación es frecuente o no. Sabemos que tanto el "Tipo de delito" como la "Descripción del lugar" son variables discretas. Por lo tanto, podemos utilizar una **tabla de contingencia** (tabla cruzada) para resumir el número total de incidentes que pertenecen a una combinación específica de valores de "Primary Type" y "Location Description".

A diferencia del análisis realizado con "Primary Type" y "Description", la `Location Description` y el `Primary Type` **NO** son variables anidadas. Por lo tanto, podemos utilizar la función `crosstab` de `pandas` para generar la tabla de contingencia de dos variables.

**Tarea 4**: La función `crosstab(var1, var2)` genera la tabla de contingencia para `var1` vs. `var2`. Utilice esta función para generar una tabla de contingencia para `Primary Type` vs `Location Description` en la que primero sólo se incluyan las 10 ubicaciones y tipos de delitos más frecuentes.

**Tarea 5**: Después use una figura de mapa de calor (Heatmap) a través del API graph_objects (go) de Plotly para visualizar la tabla de contingencia.

**Pregunta 5**: Basándose en la tabla de contingencia, ¿cuáles son los puntos calientes de los 10 tipos de delitos más frecuentes? ¿Son los mismos o no?

In [ ]:
# Coloca el código aquí ############
# Tarea 4


In [ ]:
# Coloca el código aquí ############
# Tarea 5


**RTA:**

**Pregunta 6:** Con la información que ha descubierto hasta ahora como puede desplegar los policias de Chicago.



**RTA**:

## Fase 2: Investigando crímenes a través del tiempo.

Pasamos ahora a investigar la relación entre los incidentes delictivos y el tiempo; es decir, la variable `Date`. El tiempo es una de las dimensiones más importantes para construir un plan de despliegue eficaz. Dado que no podemos patrullar todos los lugares las 24 horas del día, los 7 días de la semana, debemos centrarnos en los periodos de tiempo con altos índices de delincuencia. La "fecha" nos proporciona una marca de tiempo para cada incidente, lo que nos permite contar cuántos incidentes se produjeron en un periodo de tiempo determinado. Como disponemos de los datos de un año, podemos empezar con el total mensual de incidentes para ver si ciertos meses son propensos a la delincuencia.

Vamos a cubrir diferentes temporalidades Día, Semana y Mes para tener varías perspectivas. Tenga en cuenta que en Chicago el clima juega un papel importante.

In [ ]:
# Primero convertir la columna Date a formato fecha
df["date_py"] = pd.to_datetime(df.Date, format="mixed")

**Tarea 6:** Gráfica el número de delitos en general a través del tiempo, agrupando por mes. Puedes usar pandas para agrupar por mes y contar el número de delitos.

**Pregunta 7:** ¿Qué meses tienen tasas de delincuencia relativamente más altas?

In [ ]:
# Coloca el código aquí ############
# Tarea 6


**RTA**

**Tarea 7:** Gráfica el número de delitos en general a través del tiempo, agrupando por día. Puedes usar pandas para agrupar por día y contar el número de delitos.

**Pregunta 8:** ¿Sigue creyendo que febrero es la época en que la delincuencia es menos preocupante?

In [ ]:
# Coloca el código aquí ############
# Tarea 7


**RTA:**

**Tarea 8:** Gráfica el número de delitos en general a través del tiempo, agrupando por día de la semana. Puedes usar pandas para agrupar por día de la semana y contar el número de delitos.

**Pregunta 9:** ¿Cuáles son los días con mas y menos delitos?


In [ ]:
# Coloca el código aquí ############
# Tarea 8

**RTA**:

## Fase 3: Investigando crímenes por posición geográfica.

Otra dimensión importante que debemos considerar es la relación entre los incidentes delictivos y la ubicación geográfica. El aspecto técnico de este análisis puede verse como una profundización de lo que hemos aprendido previamente sobre visualizaciones de mapas.

Disponemos de las coordenadas geográficas aproximadas de cada incidente y, a partir de ellas, podemos explorar los patrones geográficos de la delincuencia en Chicago. Para identificar los focos geográficos de delincuencia, podemos dividir la ciudad de Chicago en regiones no superpuestas y contar el número total de casos en 2017 en cada región.En este caso, dividimos Chicago por distritos policiales y se cuentan los delitos por rondas policiales . A continuación, visualizamos los resultados en los mapas usando las librerías Plotly, Folium y geopandas:

**Pregunta 10:** Cuales puntos calientes identificaron?

Se usaron librerías adicionales ya que plotly presenta algunas complejidades y limitaciones para la visualización de mapas.

1. Con plotly se gráfica la distribución de puntos geo.

2. Se usa la librería geopandas para cargar los polígonos de las rondas policiales y cruzarlos con los puntos lon y lat y determinar asi el número de crimenes por polígono.

3. Seguidamente folium se utiliza para graficar esta información sobre un mapa de open street map, el cual es de acceso gratuito.

In [ ]:
# graficar una muestra de la distribución de puntos sobre el mapa
# de open street usando plotly para el tipo de delito mas comun
filter_df = df.loc[df['Primary Type'].isin(['THEFT', 'CRIMINAL DAMAGE'])]
#'BATTERY', 'ASSAULT', 'CRIMINAL DAMAGE']
fig = px.scatter_mapbox(filter_df, lat="Latitude",
                        lon="Longitude",
                        color='Primary Type',
                        zoom=8)

# Configurar el diseño del mapa
fig.update_layout(mapbox_style="open-street-map",
                  title='Distribución geo de delitos')

Para una funcionalidad más profunda involucrando un análisis espacial menos saturado y poder responder la pregunta 10, se usa las librerías geopandas y folium.

In [ ]:
#formatear la variable rondas policiales
df["beat_num"] = df["beat_num"].str.zfill(4)
# por rondas contar el numero de crimenes
beat_cn = df.groupby("beat_num")["ID"].count().reset_index(name="crime_count")
# Definir el esquema de color
min_cn, max_cn = beat_cn['crime_count'].quantile([0.01,0.99]).apply(round, 2)
# Crear la paleta de colores
colormap = branca.colormap.LinearColormap(
    colors=['white','yellow','orange','red','darkred'],
    #index=beat_cn['count'].quantile([0.2,0.4,0.6,0.8]),b
    vmin=min_cn,
    vmax=max_cn
)
colormap.caption="Total crimenes en Chicago por rondas policiales"

# cargar los poligonos de los sectores y limites en donde se hacen las rondas policiales
beat_orig = geopandas.read_file("./Boundaries_beat.geojson", driver = "GeoJSON")
# se interseca con los puntos
beat_data = beat_orig.join(beat_cn.set_index("beat_num"), how = "left", on = "beat_num")
beat_data.fillna(0, inplace = True)

In [ ]:
# Visualización interactiva del crimen por rondas policiales usando folium
m_crime = folium.Map(location=[41.88, -87.63],
                        zoom_start=12,
                        tiles="OpenStreetMap")
style_function = lambda x: {
    'fillColor': colormap(x['properties']['crime_count']),
    'color': 'black',
    'weight':2,
    'fillOpacity':0.5
}

stategeo = folium.GeoJson(
    beat_data.to_json(),
    name='Rondas policiales',
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(
        fields=['beat_num', 'crime_count'],
        aliases=['Ronda', 'Crimen total'],
        localize=True
    )
).add_to(m_crime)

colormap.add_to(m_crime)
m_crime

**RTA:**

**Pregunta 11**: Basándose en el análisis realizado hasta ahora, ¿cuál es su estrategia para el despliegue policial en función del tiempo, los lugares y los tipos de delitos?


**RTA:**